# NB10 — Hamming-Distance Sensitivity for Specimen Construction

**Purpose.** The paper fixes the pHash Hamming-distance threshold at 5 when clustering
near-duplicate images into specimens (Section 3.2, "Near-Duplicate Removal"). This notebook
justifies that specific value the same way the paper already justifies the 30-second
temporal-block span (Table `tab:block_sensitivity`): by sweeping the free parameter over the
**full real 11,094-image dataset**, holding the other parameter fixed at its own selected
value, and finding the largest threshold at which every one of the 11 classes still retains
at least 20 independent specimens (the minimum needed for a usable ~70/15/15 stratified
allocation).

This is **not** the synthetic 100-image ground-truth benchmark used elsewhere in the paper
(NB09) to check duplicate-detection *accuracy*. This notebook only uses the real dataset and
only asks a structural question: does this threshold leave every class with enough
independent specimens to split. It is the direct real-data counterpart to `specimen_block_sensitivity.csv`
from NB01, swept along the other axis (Hamming distance instead of block span).

**Inputs required (attach as Kaggle datasets):**
1. The raw BDLitchi image folders (11 class sub-folders of JPEGs), same as NB01/NB09.
2. `manifest_full.csv` from NB01's output (`bdlitchi-revision-splits/revision_splits/manifest_full.csv`),
   which already has `label`, `filename`, `session_id`, and `ts_eff` (the timestamp used for
   temporal blocking) for every one of the 11,094 images.

**Output:** a sensitivity table (CSV + a ready-to-paste LaTeX table) with columns matching
`tab:block_sensitivity`: threshold, specimens, mean images/specimen, smallest class, and how
many classes fall below the 20-specimen floor. A built-in check at the end confirms the
threshold=5 / span=30s row reproduces the paper's already-published 854-specimen number
exactly, as a sanity check that this notebook's Union-Find logic matches the one actually
used to build the paper's splits.


In [ ]:
# ===== Config =====
import pandas as pd
import numpy as np
import imagehash
from PIL import Image, ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True  # a couple of real JPEGs in this dataset are slightly
                                         # truncated; this lets PIL read as much as it can
                                         # instead of raising, matching what NB01 tolerates.
from pathlib import Path
import time

RAW_DATASET_DIR = '/kaggle/input/datasets/alenmaruf/bdlithi/Dataset'   # <-- EDIT if your dataset slug differs
MANIFEST_CSV = '/kaggle/input/<your-nb01-output-dataset>/manifest_full.csv'  # <-- EDIT to your NB01 output path

OUT_DIR = Path('/kaggle/working/hamming_sensitivity')
OUT_DIR.mkdir(parents=True, exist_ok=True)

BLOCK_SPAN_S = 30           # the temporal-block span already selected and used throughout the paper
THRESHOLDS = [0, 1, 2, 3, 4, 5, 6, 7, 8, 10, 12, 16, 20, 24, 32]
SPECIMEN_FLOOR = 20         # the same floor Table tab:block_sensitivity uses


In [ ]:
# ===== Load the manifest and resolve local image paths =====
df = pd.read_csv(MANIFEST_CSV)
print(f'manifest rows: {len(df)}')
assert len(df) == 11094, 'expected 11,094 rows -- check MANIFEST_CSV points at the right file'

def local_path(row):
    return Path(RAW_DATASET_DIR) / row['label'] / row['filename']

df['local_path'] = df.apply(local_path, axis=1)
missing = [p for p in df['local_path'] if not Path(p).exists()]
print(f'missing image files: {len(missing)}')
if missing:
    print('First few missing:', missing[:5])


In [ ]:
# ===== Compute pHash for every real image (same convention as NB01/NB09: imagehash.phash, 64-bit) =====
t0 = time.time()
phashes, failed = [], []
for i, p in enumerate(df['local_path']):
    try:
        phashes.append(imagehash.phash(Image.open(p)))
    except Exception as e:
        failed.append((str(p), str(e)))
        phashes.append(None)
    if (i + 1) % 2000 == 0:
        print(f'  {i+1}/{len(df)} done, {time.time()-t0:.1f}s elapsed, failures so far: {len(failed)}')
df['phash'] = phashes
print(f'pHash computed for {len(df)} images in {time.time()-t0:.1f}s; failures: {len(failed)}')
for p, e in failed:
    print('  FAILED TO READ:', p, '->', e)

# Drop any unreadable images from the sweep (should be a tiny handful at most, if any).
df_valid = df[df['phash'].notna()].reset_index(drop=True)
print(f'images used in sweep: {len(df_valid)} (dropped {len(df) - len(df_valid)} unreadable)')


In [ ]:
# ===== Union-Find, pHash edges (within-class only, as in NB01), and bounded temporal blocks =====
def popcount64(x):
    x = x - ((x >> np.uint64(1)) & np.uint64(0x5555555555555555))
    x = (x & np.uint64(0x3333333333333333)) + ((x >> np.uint64(2)) & np.uint64(0x3333333333333333))
    x = (x + (x >> np.uint64(4))) & np.uint64(0x0f0f0f0f0f0f0f0f)
    return (x * np.uint64(0x0101010101010101)) >> np.uint64(56)

hbits = np.array([int(str(h), 16) for h in df_valid['phash']], dtype=np.uint64)

class UF:
    def __init__(s, n): s.p = list(range(n))
    def find(s, x):
        while s.p[x] != x: s.p[x] = s.p[s.p[x]]; x = s.p[x]
        return x
    def union(s, a, b):
        ra, rb = s.find(a), s.find(b)
        if ra != rb: s.p[rb] = ra

# Candidate pHash pairs within each class label, distance <= 32 (well beyond anything we sweep),
# computed once and re-thresholded cheaply for every value in THRESHOLDS.
t0 = time.time()
ei, ej, ed = [], [], []
for lab, g in df_valid.groupby('label'):
    idx = g.index.to_numpy()
    H = hbits[idx]
    D = popcount64(H[:, None] ^ H[None, :])
    iu, ju = np.triu_indices(len(idx), k=1)
    dv = D[iu, ju]
    keep = dv <= 32
    ei.append(idx[iu[keep]]); ej.append(idx[ju[keep]]); ed.append(dv[keep])
edge_i, edge_j, edge_d = np.concatenate(ei), np.concatenate(ej), np.concatenate(ed)
print(f'candidate pHash edges (dist<=32): {len(edge_i)} ({time.time()-t0:.1f}s)')

# Bounded (non-transitive) temporal blocks at the fixed 30s span -- computed once, held fixed
# throughout the whole sweep, exactly mirroring how the paper holds Hamming<=5 fixed while it
# sweeps the block span in Table tab:block_sensitivity.
blk = np.empty(len(df_valid), dtype=np.int64)
b = 0
for sess, g in df_valid.groupby('session_id', sort=False):
    gi = g.index.to_numpy()
    gtm = g['ts_eff'].to_numpy()
    o = np.argsort(gtm, kind='stable')
    gi, gtm = gi[o], gtm[o]
    b += 1; start = gtm[0]
    for k in range(len(gi)):
        if gtm[k] - start > BLOCK_SPAN_S:
            b += 1; start = gtm[k]
        blk[gi[k]] = b
print('temporal blocks computed (span fixed at', BLOCK_SPAN_S, 's)')


In [ ]:
# ===== Sweep the Hamming threshold, temporal-block span held fixed at 30s =====
def build_specimens(threshold):
    uf = UF(len(df_valid))
    mask = edge_d <= threshold
    for i_, j_ in zip(edge_i[mask], edge_j[mask]):
        uf.union(int(i_), int(j_))
    o = np.argsort(blk, kind='stable'); bs = blk[o]
    starts = np.flatnonzero(np.r_[True, bs[1:] != bs[:-1]])
    for s_, e_ in zip(starts, np.r_[starts[1:], len(o)]):
        grp = o[s_:e_]
        for x in grp[1:]:
            uf.union(int(grp[0]), int(x))
    return np.array([uf.find(i_) for i_ in range(len(df_valid))])

rows = []
for T in THRESHOLDS:
    root = build_specimens(T)
    tmp = df_valid.copy()
    tmp['specimen_id'] = root
    spec_per_class = tmp.groupby('label')['specimen_id'].nunique()
    n_specimens = tmp['specimen_id'].nunique()
    worst_class = spec_per_class.idxmin()
    worst_n = int(spec_per_class.min())
    classes_under_floor = int((spec_per_class < SPECIMEN_FLOOR).sum())
    rows.append({
        'hamming_threshold': T,
        'specimens': int(n_specimens),
        'mean_imgs_per_specimen': round(len(tmp) / n_specimens, 2),
        'smallest_class': worst_class,
        'smallest_class_n': worst_n,
        'classes_under_floor': classes_under_floor,
    })
    print(rows[-1])

sweep = pd.DataFrame(rows)
sweep.to_csv(OUT_DIR / 'hamming_sensitivity.csv', index=False)
print()
print(sweep.to_string(index=False))


In [ ]:
# ===== Sanity check: threshold=5 with span=30s must reproduce the paper's published 854 specimens =====
row5 = sweep.loc[sweep.hamming_threshold == 5].iloc[0]
print(f"At Hamming<=5, span=30s: {row5.specimens} specimens "
      f"(paper reports 854 in Table tab:block_sensitivity)")
assert row5.specimens == 854, (
    'MISMATCH: this notebook does not reproduce the paper\'s published specimen count -- '
    'do not use this sweep to justify anything in the paper until this assertion passes. '
    'Check that manifest_full.csv, RAW_DATASET_DIR, and BLOCK_SPAN_S all match what NB01 used.'
)
print('OK: matches the paper exactly. The sweep above can be trusted.')


In [ ]:
# ===== Emit a LaTeX table in the same style as tab:block_sensitivity, ready to paste in =====
def fmt_thresh(lo, hi=None):
    return f'{lo}' if hi is None else f'{lo}--{hi}'

# Group consecutive identical rows for a compact table, same spirit as the manuscript table.
lines = [
    r'\begin{table}[H]',
    r'\centering',
    r'\caption{Sensitivity of the specimen count to the pHash Hamming-distance threshold, '
    r'computed once over the full 11{,}094-image manifest with the temporal-block span fixed '
    r'at 30\,s (the value selected in Section~\ref{sec:methods_split}). Hamming $\leq 5$ is '
    r"the value used throughout this paper. ``Classes $<$20 spec.'' counts classes for which "
    r'the number of independent specimens available for stratified allocation falls below 20 '
    r'at that threshold.}',
    r'\label{tab:hamming_sensitivity}',
    r'\footnotesize',
    r'\begin{tabular}{lrrlr}',
    r'\toprule',
    r'Hamming threshold & Specimens & Mean imgs/specimen & Smallest class (specimens) & Classes $<$20 spec. \\',
    r'\midrule',
]
for _, r in sweep.iterrows():
    bold = r.hamming_threshold == 5
    b0, b1 = (r'\textbf{', '}') if bold else ('', '')
    lines.append(
        f"{b0}{r.hamming_threshold}{b1} & {b0}{r.specimens}{b1} & {b0}{r.mean_imgs_per_specimen}{b1} & "
        f"{b0}{r.smallest_class} ({r.smallest_class_n}){b1} & {b0}{r.classes_under_floor}{b1} \\\\"
    )
lines += [r'\bottomrule', r'\end{tabular}', r'\end{table}']

table_tex = '\n'.join(lines)
(OUT_DIR / 'table_hamming_sensitivity.tex').write_text(table_tex)
print(table_tex)
